In [30]:
%pip install -q pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import pandas as pd

from src.constants import INPUT_DIR, OUTPUT_DIR, Column

In [32]:
meta_file = INPUT_DIR / "meta_Electronics.jsonl"
output_file = OUTPUT_DIR / "monitor_meta.csv"

def is_pc_monitor(categories):
    return (
        "Computers & Accessories" in categories
        and "Monitors" in categories
    )


columns_needed = [
    "parent_asin",
    "title",
    "main_category",
    "average_rating",
    "rating_number",
    "price",
    "store",
    "categories",
    "details"
]

first_chunk = True

for chunk in pd.read_json(
    meta_file,
    lines=True,
    chunksize=100_000
):
    # Nur benötigte Spalten behalten
    chunk = chunk[columns_needed]

    # Nur PC-Monitore behalten
    mask = chunk["categories"].apply(is_pc_monitor)
    chunk = chunk[mask]

    # Stückweise an CSV anhängen
    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

In [33]:
import pandas as pd
import ast

# CSV einlesen
pc_monitors = pd.read_csv(
    OUTPUT_DIR / "monitor_meta.csv"
)

exclude_terms = [
    "raspberry pi",
    "raspb pi",
    "rpi",
    "temperature",
    "temp monitor",
    "repair kit",
    "controller board",
    "arcade machine",
    "game hat",
    "lcd module",
    "car monitor",
    "gauge cluster",
    "fitness tracker",
    "heart rate",
    "screen protector"
]

title_lower = pc_monitors["title"].fillna("").str.lower()

exclude_mask = False

for term in exclude_terms:
    exclude_mask = (
        exclude_mask
        | title_lower.str.contains(term, regex=False)
    )

pc_monitors_clean = pc_monitors[~exclude_mask].copy()

total_ratings = pc_monitors_clean["rating_number"].sum()

print("Produkte:", len(pc_monitors_clean))
print("Summe rating_number:", total_ratings)

print(
    pc_monitors_clean["rating_number"].describe(
        percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

Produkte: 9292
Summe rating_number: 1610820
count     9292.000000
mean       173.355575
std       1020.378359
min          1.000000
25%          3.000000
50%         13.000000
75%         57.250000
90%        258.900000
95%        676.450000
99%       2931.090000
max      40868.000000
Name: rating_number, dtype: float64


In [34]:
monitor_asins = set(
    pc_monitors_clean["parent_asin"]
    .dropna()
    .unique()
)

review_file =  INPUT_DIR / "Electronics.jsonl"
output_file = OUTPUT_DIR / "monitor_reviews.csv"

chunk_number = 0
total_reviews = 0
first_chunk = True

for chunk in pd.read_json(
    review_file,
    lines=True,
    chunksize=100_000
):
    chunk_number += 1

    # Nur Reviews von PC-Monitoren
    chunk = chunk[
        chunk["parent_asin"].isin(monitor_asins)
    ]

    total_reviews += len(chunk)

    if not chunk.empty:
        chunk.to_csv(
            output_file,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False

    if chunk_number % 10 == 0:
        print(
            f"Chunk {chunk_number} verarbeitet | "
            f"Gefundene Reviews: {total_reviews}"
        )

print("Fertig.")
print("Insgesamt gefundene Monitor-Reviews:", total_reviews)

Chunk 10 verarbeitet | Gefundene Reviews: 6566
Chunk 20 verarbeitet | Gefundene Reviews: 13390
Chunk 30 verarbeitet | Gefundene Reviews: 19854
Chunk 40 verarbeitet | Gefundene Reviews: 26821
Chunk 50 verarbeitet | Gefundene Reviews: 33213
Chunk 60 verarbeitet | Gefundene Reviews: 39723
Chunk 70 verarbeitet | Gefundene Reviews: 46562
Chunk 80 verarbeitet | Gefundene Reviews: 53251
Chunk 90 verarbeitet | Gefundene Reviews: 59690
Chunk 100 verarbeitet | Gefundene Reviews: 65896
Chunk 110 verarbeitet | Gefundene Reviews: 72094
Chunk 120 verarbeitet | Gefundene Reviews: 78547
Chunk 130 verarbeitet | Gefundene Reviews: 85625
Chunk 140 verarbeitet | Gefundene Reviews: 92203
Chunk 150 verarbeitet | Gefundene Reviews: 98770
Chunk 160 verarbeitet | Gefundene Reviews: 105759
Chunk 170 verarbeitet | Gefundene Reviews: 112496
Chunk 180 verarbeitet | Gefundene Reviews: 119280
Chunk 190 verarbeitet | Gefundene Reviews: 126071
Chunk 200 verarbeitet | Gefundene Reviews: 132845
Chunk 210 verarbeitet | G

In [46]:
monitor_reviews = pd.read_csv(output_file)

monitor_reviews_sample_raw = (
    monitor_reviews
    .sample(
        n=10_000,
        random_state=42
    )
    .reset_index(drop=True)
)

monitor_sample = pd.DataFrame()

monitor_sample[Column.ID] = range(len(monitor_reviews_sample_raw))
monitor_sample[Column.TEXT] = monitor_reviews_sample_raw["text"]
monitor_sample[Column.RATING] = monitor_reviews_sample_raw["rating"]

monitor_sample.to_csv(
    INPUT_DIR / "monitor_reviews_sample_10000.csv",
    index=False
)


